In [1]:

from jupyter_lab_notebook_toc_utils import generate_toc, display_toc
toc = generate_toc(add_numbering=True)
display_toc(toc)

**Table of Contents**<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**1.** **Overview**](#Overview)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**2.** **Licensing and Commercial Support**](#Licensing-and-Commercial-Support)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**3.** **Solution Architecture**](#Solution-Architecture)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.1.** Backend](#Backend)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.2.** Frontend](#Frontend)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.3.** Database](#Database)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.4.** Scalability](#Scalability)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.5.** RBAC & Security Features](#RBAC-&-Security-Features)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.6.** VCS Integration](#VCS-Integration)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.7.** Portability](#Portability)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.8.** REST API](#REST-API)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**3.9.** External Hooks](#External-Hooks)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**4.** **Major Concepts**](#Major-Concepts)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.1.** Workflow](#Workflow)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.2.** Workflow Templates](#Workflow-Templates)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.3.** Nodes](#Nodes)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.3.1.** LangChain Integration](#LangChain-Integration)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.4.** Connections](#Connections)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.5.** Logical Operators](#Logical-Operators)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.6.** Sticky Notes](#Sticky-Notes)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.7.** Tags](#Tags)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.8.** Credentials](#Credentials)<br/>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; [**4.9.** Users & Account Types](#Users-&-Account-Types)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**5.** **Installation**](#Installation)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**6.** **Install debugging utilities**](#Install-debugging-utilities)<br/>
&nbsp;&nbsp;&nbsp;&nbsp; [**7.** **Install node and required packages**](#Install-node-and-required-packages)

# Overview

The n8n (pronounced n-eight-n) solution is a workflow automation tool with native support for AI. Like many of it's competitors, it represents workflows as as graphs which are configured through a no or low code drag and drop canvas. It is provided implemented as a node.js executable and does have the ability to horizontally scale.

# Licensing and Commercial Support

After looking at the site, it appears there are several tiers of products being offered:
* Comunity Edition - A "source available" basic version provided on github
* A Hosted Solution (with starter, pro, and enterprise subscriptions) - offering varying SLAs and support.
* A Self-hosted Enterprise Subscription
* A Startup Plan - Avaiable to startups with up to 20 employees which raised up to $5M.

According to the [LICENSE.md](https://github.com/n8n-io/n8n/blob/master/LICENSE.md) file in the github repository, the source code is distributed under several licences:
* Content of branches other than the main branch (i.e. "master") are not licensed.
* All source code files that contain ".ee." in their filename are licensed under the "n8n Enterprise License" defined in "LICENSE_EE.md".
* All third party components incorporated into the n8n Software are licensed under the original license provided by the owner of the applicable component.
* Content outside of the above mentioned files or restrictions is available under the "Sustainable Use License" (defined within this file)

# Solution Architecture

## Backend
The n8n solution is provided as an executable which runs on node.js. As such it is packaged and distributed by npm and npx. The official github page can be found [here](https://github.com/n8n-io/n8n).

## Frontend
The frontend is implemented as a ReactJS application. As all of the Workflows and Nodes are defined as TypeScript classes, they are orchestrated and executed within the singular binary. 

## Database

n8n uses a database to save credentials, past executions, and workflows. By default, n8n uses SQLite, however n8n also supports using an external PostgresDB installation.

## Scalability

While the solution is deployed as a single executable with a dependency on a database, it is possible to scale the solution with some engineering work.

The first thing to do would be to transition from the default sqllite database to a scalable postgresql cluster.

Next would be to implement a master-slave configuration called "queue-mode" as described in the [documentation](https://docs.n8n.io/hosting/scaling/queue-mode/):

> When running in queue mode, you have multiple n8n instances set up, with one main instance receiving workflow information (such as triggers) and the worker instances performing the executions.
>
> Each worker is its own Node.js instance, running in main mode, but able to handle multiple simultaneous workflow executions due to their high IOPS (input-output operations per second).
>
> By using worker instances and running in queue mode, you can scale n8n up (by adding workers) and down (by removing workers) as needed to handle the workload.

This capability requires additional architectural components; a Redis database maintains the work queue and a database (such as postgres) records the results. The master and slaves communicate through these two data stores. 

n8n does not provide mechanisms for deploying this configuration or the required datastores; that is left to the engineer.

## RBAC & Security Features

The solution offers two factor authentication, LDAP integration, and SAML SSO. However this is only available to the paid licenses. For more details consult the [documentation](https://docs.n8n.io/user-management/).

Within n8n there are three [acount types](https://docs.n8n.io/user-management/account-types/): 
- **Owner**: this is the account that set up user management. There's one owner account for each n8n instance.
    - Add and remove users, including admin users
    - Upgrade members to admin, and downgrade admins to member
    - See and share all workflows
    - See, edit, and share all credentials (but not see the sensitive information)
    - Delete tags
    - Set up and use Source control
- **Admin**: elevated permissions within the app. An admin can do everything that an owner can, except:
    - Access the Cloud dashboard
    - Modify the owner or change the owner role
- **Members**: these are normal n8n users. Members can:
    - See all workflow tags, create new tags, and assign tags to their workflows. Members can't delete tags.
    - Change their own password.
    - Change their own email.
    - See their own workflows.

## VCS Integration

Git is integrated directly into n8n as decribed [here](https://docs.n8n.io/source-control-environments/understand/git/), however as noted in the documentation:

> n8n doesn't implement all Git functionality: you shouldn't view n8n's source control as full version control.

**Note**: The integration is only available to those who purchased the enterprise plan.

This integration allows users to edit their workflows in the n8n web interface while having them versioned and stored in a git provider. It appears that n8n allows for committing, pushing and pulling. The rest of the git workflow will need to be executed on the backend VCS.

Looking at the [screenshots](https://docs.n8n.io/source-control-environments/using/push-pull), it appears that the UI provides built in controls for pushing and pulling a workflow to a branch. However, n8n recommends only establishing a unidirectional workflow:

> You can push work from an instance to a branch, and pull to the same instance. n8n doesn't recommend this. To reduce the risk of merge conflicts and overwriting work, try to create a process where work goes in one direction: either to Git, or from Git, but not both.



Another important consideration when using the git integration is the effect the push/pull events have on the workflow:

> If you pull changes to an active workflow, n8n sets the workflow to inactive while pulling, then reactivates it. This may result in a few seconds of downtime for the workflow.

Aside from this integration, a manual workflow is possible; users can import/export Workflows and other configurations as JSON documents and then manually commit/copy them from the git repository.

## Portability

As mentioned above, the flows can be imported and exported as JSON through the UI. The n8n solution does not offer an SDK for managing Workflows through code.

## REST API

While the documentation mentions there is a REST API, it is not clear if this is feature complete (i.e. anything possible in the UI is possible via the API). Looking through the [documentation](https://docs.n8n.io/api) it appears that creating Users and Workflows is possible as well as triggering a workflow.

## External Hooks

We [see](https://docs.n8n.io/embed/configuration/#frontend-external-hooks) that n8n allows us to execute webhooks when specific actions occur on the frontend and the backend:

> Like backend external hooks, it's possible to define external hooks in the frontend code that get executed by n8n whenever a specific operation is performed. They can be used, for example, to log data and change data.

# Major Concepts

## Workflow

A workflow is a logical collection of related Nodes. Conceptually, the workflow describes an end-to-end process that is articulated by one or more Nodes. The Workflows are implimented as graphs of Nodes and visualized on a drag-and-drop canvas.

## Workflow Templates

As the name suggests, a Workflow template is a reusable workflow definition, which consists of a preconfigured set of Nodes which a user can instantiate and configure to produce a fully functional workflow without having to design it from scratch. 

The [integration page](https://n8n.io/integrations/) lists a number of example templates which can be downloaded and installed.

## Nodes

According to the [documentation ](https://docs.n8n.io/workflows/components/nodes), Nodes are the key building blocks of a workflow. The Nodes divide the workflow into smaller pieces and abstract away the underlying implementation details. n8n provides a collection of built-in nodes, as well as the ability to download and install comunity nodes, and create your own nodes. 

Nodes are implemented as TypeScript classes which extend the INodeType interface. There are two styles (declarative and programatic) for defining a node as described [here](https://docs.n8n.io/integrations/creating-nodes/plan/choose-node-method/#syntax-differences). Additionally, n8n provides libraries for defining the UI elements of the Nodes.

As the nodes are defined as TypeScript classes, they can be packaged and distributed through npm as described [here](https://www.npmjs.com/package/n8n-node-dev).

### LangChain Integration

The Nodes provide the user with the ability to leverage LangChain functionality. As the Nodes are implimented in javscript, the LangChain integration happens through the javascript LangChain library. There are several Nodes which n8n offers to expose LangChain functionality, for more information review this [documentation](https://docs.n8n.io/advanced-ai/langchain/langchain-n8n/).

## Connections

According to the [documentation](https://docs.n8n.io/workflows/components/connections/):

> A connection establishes a link between nodes to route data through the workflow. A connection between two nodes passes data from one node's output to another node's input.

## Logical Operators

The n8n solution offers a number of [means for expressing logical patterns](https://docs.n8n.io/flow-logic/) within a Workflow. Conditional splitting, Looping, Waiting, Error Handling, and more are all provided as off-the-shelf functionality which is exposed via the Nodes.

## Sticky Notes

According to the [documentation](https://docs.n8n.io/workflows/components/sticky-notes/), Sticky Notes allow you to annotate and comment on your workflows by visually placing a colored box of text within the workflow canvas. Sticky Notes support the use of Markdown.

## Tags

Within n8n, [tags](https://docs.n8n.io/workflows/tags/) allow the user to attach globally available labels to workflows. These labels can then be used when querying the system to filter based on the Tag.

## Credentials

[Credentials](https://docs.n8n.io/credentials/) allow n8n users to store the various bits of information used to authenticate with external systems. Users can create and share credentials with other users. The credentials can be linked to Nodes, giving the node the ability to authenticate while interacting with an external system.

## Users & Account Types

When a user is created, the User is assigned an account type. There are three account types, owner, admin, and member. The account type affects the user permissions and access.For more information on account types, see this [documentation](https://docs.n8n.io/user-management/account-types/).

# Installation

According to the [github page](https://github.com/n8n-io/n8n/tags) the latest version of n8n is 1.38.1 however the latest version slated for production use is 1.37.3 as noted [here](https://docs.n8n.io/hosting/installation/npm/). Reading through the documentation, it appears the solution is packaged and deployable via the following two mechanisms:

- npm Package
- Prebuilt docker container

These in turn will allow the solution to be deployed to cloud providers or platforms like kubernetes.

In my case, I will build my own docker container for a quick test of the npm package. Below are the instructions for building the docker container

```
(base) [root@fedora n8n-test]# cat Dockerfile
FROM python:3.11.3-bullseye

# Install debugging utilities

RUN apt-get -y update
RUN apt-get -y install iproute2 net-tools traceroute vim

# Install node and required packages

RUN apt-get -y update
RUN curl -sL https://deb.nodesource.com/setup_18.x | bash -
RUN apt-get install -y nodejs
RUN node --version
RUN npm --version
(base) [root@fedora n8n-test]#  docker build -t n8n-test .
```

And here are the commands I used to run the container, then execute the command to install inside the container, and finally run an instance of n8n.

```
(base) [root@fedora n8n-test]#  docker run -ti --net=host n8n-test /bin/bash
root@fedora:/# node --version
v18.20.2
root@fedora:/# export N8N_HOST=0.0.0.0
root@fedora:/# export N8N_PROTOCOL=http
root@fedora:/# export N8N_SECURE_COOKIE=false
root@fedora:/# npx n8n
```

**Note**: The environment variables being set were recommended in this [thread](https://community.n8n.io/t/change-default-locahost-to-0-0-0-0/40245/2).